In [ ]:
"""
RDD2022 Country-Aware Class Balancing Script
=============================================
Goal: Create RDD2022_Balanced by oversampling minority classes per country.

BEFORE RUNNING:
  1. Set DATASET_ROOT to your RDD2022 path
  2. Set OUTPUT_ROOT to where you want the balanced dataset saved
  3. Make sure your label files are YOLO .txt format (class x y w h per line)
     If you still have XML, run the XML->YOLO conversion first (see bottom of file)

FOLDER STRUCTURE EXPECTED:
  DATASET_ROOT/
    India/
      train/
        images/   <- .jpg files
        labels/   <- .txt files (YOLO format)
    Japan/
      train/
        images/
        labels/
    ... etc

OUTPUT:
  OUTPUT_ROOT/
    India/
      train/
        images/   <- originals + augmented copies
        labels/   <- corresponding label files
    ... etc
"""

import os
import shutil
import random
import math
from PIL import Image, ImageEnhance, ImageFilter
import numpy as np
from collections import defaultdict

# ============================================================
# SET THESE TWO PATHS BEFORE RUNNING
# ============================================================
DATASET_ROOT = "/kaggle/input/YOUR_DATASET_NAME"   # <-- change this
OUTPUT_ROOT  = "/kaggle/working/RDD2022_Balanced"  # <-- this is fine as-is
# ============================================================

random.seed(42)
np.random.seed(42)

# Class index mapping (YOLO format)
CLASS_NAMES = {0: "D00", 1: "D10", 2: "D20", 3: "D40"}
CLASS_IDX   = {"D00": 0,  "D10": 1,  "D20": 2,  "D40": 3}

# ============================================================
# BALANCING TARGETS
# Rule: if ratio > 10:1, increase minority until ratio ~5:1
# Japan is left as-is (ratio only 2.8:1, not worth the risk)
# ============================================================
# Format: country -> {class_to_increase: target_instance_count}
BALANCE_CONFIG = {
    "India":           {"D10": 600},
    "United_States":   {"D40": 1000},
    "Czech":           {"D20": 300},
    "China_Drone":     {"D40": 280},
    "China_MotorBike": {"D40": 480},
    "Norway":          {"D40": 400},
    # Japan: skipped intentionally (imbalance ratio ~2.8:1)
}


# ============================================================
# STEP 1: COUNT LABELS
# ============================================================

def count_labels_in_dir(labels_dir):
    """Count instances of each class in a labels directory."""
    counts = defaultdict(int)
    if not os.path.exists(labels_dir):
        return counts
    for fname in os.listdir(labels_dir):
        if not fname.endswith(".txt"):
            continue
        with open(os.path.join(labels_dir, fname)) as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                cls = int(line.split()[0])
                counts[cls] += 1
    return counts


def get_images_containing_class(labels_dir, class_idx):
    """Return list of image basenames (no extension) that contain the given class."""
    result = []
    if not os.path.exists(labels_dir):
        return result
    for fname in os.listdir(labels_dir):
        if not fname.endswith(".txt"):
            continue
        with open(os.path.join(labels_dir, fname)) as f:
            classes_in_image = [int(l.split()[0]) for l in f if l.strip()]
        if class_idx in classes_in_image:
            result.append(fname.replace(".txt", ""))
    return result


# ============================================================
# STEP 2: AUGMENTATION FUNCTIONS
# ============================================================

def aug_brightness(img):
    factor = random.uniform(0.6, 1.4)
    return ImageEnhance.Brightness(img).enhance(factor)

def aug_contrast(img):
    factor = random.uniform(0.6, 1.4)
    return ImageEnhance.Contrast(img).enhance(factor)

def aug_blur(img):
    return img.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.5, 1.5)))

def aug_noise(img):
    arr = np.array(img).astype(np.int16)
    noise = np.random.randint(-25, 25, arr.shape)
    arr = np.clip(arr + noise, 0, 255).astype(np.uint8)
    return Image.fromarray(arr)

def aug_sharpness(img):
    factor = random.uniform(0.3, 2.5)
    return ImageEnhance.Sharpness(img).enhance(factor)

def aug_flip(img):
    """Horizontal flip. Returns (flipped_image, needs_bbox_update=True)"""
    return img.transpose(Image.FLIP_LEFT_RIGHT), True

def flip_labels(label_lines):
    """Update bounding boxes after horizontal flip: new_x = 1 - old_x"""
    new_lines = []
    for line in label_lines:
        parts = line.strip().split()
        if len(parts) != 5:
            new_lines.append(line)
            continue
        cls, x, y, w, h = parts
        new_x = 1.0 - float(x)
        new_lines.append(f"{cls} {new_x:.6f} {y} {w} {h}")
    return new_lines


# Pool of augmentations: (function, flips_bbox)
AUGMENTATIONS = [
    ("brightness", aug_brightness, False),
    ("contrast",   aug_contrast,   False),
    ("blur",       aug_blur,       False),
    ("noise",      aug_noise,      False),
    ("sharpness",  aug_sharpness,  False),
    ("flip",       aug_flip,       True),   # flip handled separately
]


def apply_augmentation(img, aug_name, label_lines):
    """
    Apply a named augmentation to image and update labels if needed.
    Returns (augmented_image, updated_label_lines).
    """
    for name, fn, flips in AUGMENTATIONS:
        if name != aug_name:
            continue
        if flips:
            aug_img, _ = fn(img)
            new_labels = flip_labels(label_lines)
        else:
            aug_img = fn(img)
            new_labels = label_lines
        return aug_img, new_labels
    raise ValueError(f"Unknown augmentation: {aug_name}")


def pick_augmentation(aug_id):
    """Cycle through augmentations deterministically."""
    aug_names = [a[0] for a in AUGMENTATIONS]
    return aug_names[aug_id % len(aug_names)]


# ============================================================
# STEP 3: COPY ORIGINALS (without duplicating the images on disk)
# ============================================================

def copy_originals(src_country_path, dst_country_path):
    """
    Copy original images + labels into output folder.
    We use symlinks where possible to save disk space,
    but fall back to actual copy if symlinks aren't supported.
    """
    for split in ["train"]:
        src_img = os.path.join(src_country_path, split, "images")
        src_lbl = os.path.join(src_country_path, split, "labels")
        dst_img = os.path.join(dst_country_path, split, "images")
        dst_lbl = os.path.join(dst_country_path, split, "labels")

        os.makedirs(dst_img, exist_ok=True)
        os.makedirs(dst_lbl, exist_ok=True)

        if os.path.exists(src_img):
            for f in os.listdir(src_img):
                shutil.copy2(os.path.join(src_img, f), os.path.join(dst_img, f))

        if os.path.exists(src_lbl):
            for f in os.listdir(src_lbl):
                shutil.copy2(os.path.join(src_lbl, f), os.path.join(dst_lbl, f))

    print(f"  Copied originals.")


# ============================================================
# STEP 4: OVERSAMPLE MINORITY CLASS
# ============================================================

def oversample_class(country_name, dst_country_path, src_labels_dir, src_images_dir,
                     class_name, target_count, current_count):
    """
    Generate augmented copies of minority class images until target_count is reached.
    Saves to dst_country_path/train/images and dst_country_path/train/labels.
    """
    class_idx = CLASS_IDX[class_name]
    minority_basenames = get_images_containing_class(src_labels_dir, class_idx)

    if not minority_basenames:
        print(f"  WARNING: No images found containing {class_name} in {country_name}. Skipping.")
        return

    needed = target_count - current_count
    if needed <= 0:
        print(f"  {class_name} already at or above target ({current_count} >= {target_count}). Skipping.")
        return

    print(f"  {class_name}: {current_count} -> target {target_count} (need ~{needed} more instances)")

    dst_img_dir = os.path.join(dst_country_path, "train", "images")
    dst_lbl_dir = os.path.join(dst_country_path, "train", "labels")

    aug_id = 0
    added_instances = 0

    while added_instances < needed:
        # Pick a random source image containing this class
        basename = random.choice(minority_basenames)
        src_img_path = os.path.join(src_images_dir, basename + ".jpg")
        src_lbl_path = os.path.join(src_labels_dir, basename + ".txt")

        # Skip if files missing
        if not os.path.exists(src_img_path) or not os.path.exists(src_lbl_path):
            aug_id += 1
            continue

        # Read label file
        with open(src_lbl_path) as f:
            label_lines = f.read().strip().split("\n")

        # Count how many instances of target class are in this image
        instance_count = sum(1 for l in label_lines if l.strip() and int(l.split()[0]) == class_idx)

        # Pick augmentation
        aug_name = pick_augmentation(aug_id)

        # Generate new filenames
        new_basename = f"aug_{country_name}_{class_name}_{aug_id:05d}"
        new_img_path = os.path.join(dst_img_dir, new_basename + ".jpg")
        new_lbl_path = os.path.join(dst_lbl_dir, new_basename + ".txt")

        # Apply augmentation
        try:
            img = Image.open(src_img_path).convert("RGB")
            aug_img, new_labels = apply_augmentation(img, aug_name, label_lines)
            aug_img.save(new_img_path, quality=95)
            with open(new_lbl_path, "w") as f:
                f.write("\n".join(new_labels))
        except Exception as e:
            print(f"  ERROR on {basename} aug {aug_name}: {e}")
            aug_id += 1
            continue

        added_instances += instance_count
        aug_id += 1

    print(f"  Done. Generated {aug_id} augmented images for {class_name}.")


# ============================================================
# STEP 5: VERIFY FINAL DISTRIBUTION
# ============================================================

def verify_distribution(output_root, countries):
    """Print before/after class distribution for each country."""
    print("\n" + "="*60)
    print("VERIFICATION: Final class distribution")
    print("="*60)
    print(f"{'Country':<20} {'D00':>6} {'D10':>6} {'D20':>6} {'D40':>6}")
    print("-"*50)

    for country in countries:
        lbl_dir = os.path.join(output_root, country, "train", "labels")
        counts = count_labels_in_dir(lbl_dir)
        d00 = counts.get(0, 0)
        d10 = counts.get(1, 0)
        d20 = counts.get(2, 0)
        d40 = counts.get(3, 0)
        print(f"{country:<20} {d00:>6} {d10:>6} {d20:>6} {d40:>6}")

    print("="*60)


# ============================================================
# STEP 6: MAIN
# ============================================================

def main():
    os.makedirs(OUTPUT_ROOT, exist_ok=True)

    # --- Print BEFORE distribution ---
    print("\n" + "="*60)
    print("BEFORE: Original class distribution")
    print("="*60)
    print(f"{'Country':<20} {'D00':>6} {'D10':>6} {'D20':>6} {'D40':>6}")
    print("-"*50)

    all_countries = list(BALANCE_CONFIG.keys()) + ["Japan"]
    for country in all_countries:
        lbl_dir = os.path.join(DATASET_ROOT, country, "train", "labels")
        counts = count_labels_in_dir(lbl_dir)
        print(f"{country:<20} {counts.get(0,0):>6} {counts.get(1,0):>6} {counts.get(2,0):>6} {counts.get(3,0):>6}")
    print("="*60)

    # --- Process each country ---
    for country, class_targets in BALANCE_CONFIG.items():
        print(f"\n{'='*40}")
        print(f"Processing: {country}")
        print(f"{'='*40}")

        src_country = os.path.join(DATASET_ROOT, country)
        dst_country = os.path.join(OUTPUT_ROOT, country)

        src_img_dir = os.path.join(src_country, "train", "images")
        src_lbl_dir = os.path.join(src_country, "train", "labels")

        if not os.path.exists(src_img_dir):
            print(f"  WARNING: {src_img_dir} not found. Skipping.")
            continue

        # Copy originals first
        copy_originals(src_country, dst_country)

        # Get current counts
        current_counts = count_labels_in_dir(src_lbl_dir)

        # Oversample each minority class
        for class_name, target in class_targets.items():
            class_idx = CLASS_IDX[class_name]
            current = current_counts.get(class_idx, 0)
            oversample_class(
                country_name=country,
                dst_country_path=dst_country,
                src_labels_dir=src_lbl_dir,
                src_images_dir=src_img_dir,
                class_name=class_name,
                target_count=target,
                current_count=current,
            )

    # --- Copy Japan as-is (no balancing needed) ---
    print(f"\n{'='*40}")
    print("Processing: Japan (no balancing, copying as-is)")
    print(f"{'='*40}")
    copy_originals(
        os.path.join(DATASET_ROOT, "Japan"),
        os.path.join(OUTPUT_ROOT, "Japan")
    )

    # --- Print AFTER distribution ---
    verify_distribution(OUTPUT_ROOT, all_countries)

    print("\nDone. Balanced dataset saved to:", OUTPUT_ROOT)


if __name__ == "__main__":
    main()


# ============================================================
# BONUS: XML -> YOLO CONVERSION
# Run this FIRST if your labels are still in XML format.
# ============================================================

def convert_xml_to_yolo(dataset_root, countries):
    """
    Convert Pascal VOC XML annotations to YOLO .txt format.
    Creates a 'labels' folder next to the 'annotations/xmls' folder.
    """
    import xml.etree.ElementTree as ET

    for country in countries:
        xml_dir = os.path.join(dataset_root, country, "train", "annotations", "xmls")
        lbl_dir = os.path.join(dataset_root, country, "train", "labels")

        if not os.path.exists(xml_dir):
            print(f"Skipping {country}: no xmls folder found")
            continue

        os.makedirs(lbl_dir, exist_ok=True)
        xml_files = [f for f in os.listdir(xml_dir) if f.endswith(".xml")]
        print(f"Converting {len(xml_files)} XMLs for {country}...")

        for xml_file in xml_files:
            tree = ET.parse(os.path.join(xml_dir, xml_file))
            root = tree.getroot()

            size = root.find("size")
            W = int(float(size.find("width").text))
            H = int(float(size.find("height").text))

            lines = []
            for obj in root.findall("object"):
                name = obj.find("name").text
                if name not in CLASS_IDX:
                    continue
                cls = CLASS_IDX[name]
                bb = obj.find("bndbox")
                xmin = float(bb.find("xmin").text)
                ymin = float(bb.find("ymin").text)
                xmax = float(bb.find("xmax").text)
                ymax = float(bb.find("ymax").text)

                x_center = (xmin + xmax) / (2 * W)
                y_center = (ymin + ymax) / (2 * H)
                width    = (xmax - xmin) / W
                height   = (ymax - ymin) / H

                lines.append(f"{cls} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}")

            out_file = os.path.join(lbl_dir, xml_file.replace(".xml", ".txt"))
            with open(out_file, "w") as f:
                f.write("\n".join(lines))

        print(f"  Done: {country}")

# To run conversion, call:
# convert_xml_to_yolo(DATASET_ROOT, ["India", "Japan", "Czech", "Norway", "United_States", "China_Drone", "China_MotorBike"])